# Lesson 7.3 · Trees in practice

Regression, feature importances and their pitfalls, categoricals and missing values, and the limits of a model with axis-parallel splits.

**What you will learn**

- Regression trees: piecewise constant predictions, and why they do not extrapolate.
- Importances: the impurity-based one (`feature_importances_`) and the permutation one, with their biases.
- What trees do not do well.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

## Regression: piecewise constant

`DecisionTreeRegressor` splits by minimizing the squared error and predicts the **mean of the target** in the leaf. The result is a step function: it can follow nonlinear relationships, but with few examples per leaf it is noisy and **outside the training range it is flat**.

In [ ]:
rng = np.random.default_rng(0)
x = np.sort(rng.uniform(0, 6, 200)); y = np.sin(x) + 0.3 * x + rng.normal(0, 0.25, 200)
grid = np.linspace(-1, 8, 400).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.scatter(x, y, s=10, color="gray")
for d in [2, 4, None]:
    ax.step(grid, DecisionTreeRegressor(max_depth=d, min_samples_leaf=5, random_state=0).fit(x.reshape(-1, 1), y).predict(grid), where="mid", label=f"max_depth={d}")
ax.axvspan(6, 8, color="orange", alpha=0.1); ax.legend(); plt.show()

Beyond the value 6 the prediction is a **constant**: the tree does not continue the trend (like KNN, lesson 5.3, and unlike a linear regression). It is a serious limit with data that change over time (exercise C1).

## Feature importance

Every split reduces the impurity: summing the reductions per feature (weighted by the number of examples) gives `feature_importances_`. It is free and convenient, but it has a flaw: it **favors features with many distinct values**, even useless ones, because they have more thresholds to try and more chances of finding a lucky one.

In [ ]:
rng = np.random.default_rng(1)
n = 800
signal = rng.normal(size=(n, 2))
y = (signal[:, 0] + 0.5 * signal[:, 1] + rng.normal(0, 1, n) > 0).astype(int)
continuous_noise = rng.normal(size=n)                         # many distinct values, no information
binary_noise = rng.integers(0, 2, n)                          # 2 values, no information
X = np.c_[signal, continuous_noise, binary_noise]
names = ["signal 1", "signal 2", "continuous noise", "binary noise"]
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.4, random_state=0)
t = DecisionTreeClassifier(min_samples_leaf=3, random_state=0).fit(Xa, ya)
perm = permutation_importance(t, Xb, yb, n_repeats=20, random_state=0)
print(pd.DataFrame({"importance (impurity)": t.feature_importances_, "permutation (test)": perm.importances_mean}, index=names).round(3))

The impurity importance gives the **continuous noise** 24% (more than the second signal, 21%), while the permutation on the test set (which measures how much the accuracy gets worse by shuffling that column) brings it to about zero. Rule: to judge the usefulness of a feature use the permutation importance **on data not used
for training** (exercise D2).

## Limits

- **Axis-parallel splits:** a diagonal boundary (`x1 > x2`) needs a staircase; a logistic regression describes it with a single coefficient (exercise F2).
- **Instability**: see above.
- **No extrapolation** in regression problems.
- **Missing values:** in scikit-learn 1.1 trees do **not** accept NaN (from 1.3 they do): you need an imputer (with a missingness indicator, as in module 4).
- **Categoricals:** one-hot works but fragments; the ordinal encoding (`OrdinalEncoder`) lets the tree treat the categories as numbers and split them with a threshold: with many levels it is often better.

> **Common mistake.** Reading `feature_importances_` as "the effect" of a variable: it is a measure of how much the tree used it, not of how much the variable matters in the world, and with correlated features the importance is split arbitrarily.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>Why does a tree not need scaling but logistic regression does?</summary>

Because the tree only uses the order of the values of each feature (thresholds), which does not change under increasing monotonic transformations. Logistic regression multiplies the values by coefficients and sums them: the scale determines the relative weight (and the penalty), and it changes the model.

</details>

<details><summary>When do you prefer a single tree to a forest?</summary>

When explainability matters most (written rules, audits) or you need a very small model that is fast to implement. For performance a forest or a boosting almost always wins, at the price of losing the direct reading of the rules.

</details>